# autoguidance — 10 · Experiment Runner  (RUN OFFLINE / Internet = OFF)

Runs **Phase 0** (weak-self characterization) and optionally **Phase 1** (baseline
decoders × NFE ladder) on a single **RTX PRO 6000 (96 GB)** in **bf16 on cuda:0**.

**Before you run:** attach the Kaggle Datasets produced by `00_dataset_builder.ipynb`
(LLaDA *or* DiffusionGemma weights, gpt2-large, mauve, wheels, hfdata, code) and set
**Internet = OFF**, **Accelerator = GPU**.

Flow: config → GPU probe → mount check → offline pip → install code → extract weights
→ **synthetic dry-run** → load real adapter → (Gemma) MoE patch check → Phase 0
characterize → cheap threshold/sweep cell → Phase 0 report → Phase 1 → egress.

> Pick **one** model per session. `MODEL="llada"` uses the `tf446` wheels;
> `MODEL="diffusiongemma"` uses `tf5`. Never extract both weight tars (~70 GB temp).


## 1 · CONFIG — mounts, model choice, toggles, HF-offline env

In [ ]:
# ----------------------------------------------------------------------------------
# SINGLE SOURCE OF TRUTH for the offline run. HF offline env vars are set HERE,
# before ANY transformers/datasets import anywhere in the notebook.
# ----------------------------------------------------------------------------------
import os

# >>> set offline BEFORE importing transformers/datasets/huggingface_hub <<<
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
# WSL2/discrete-GPU allocator note carried over for safety; harmless on Kaggle.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# ----- which model this session runs (pick ONE) ----------------------------------
MODEL = "llada"                 # "llada" | "diffusiongemma"
# variant selects the wheelhouse + must match what the weights need:
#   llada -> tf446 (transformers==4.46.3) ; diffusiongemma -> tf5 (transformers>=5.12)
TRANSFORMERS_VARIANT = "tf446" if MODEL == "llada" else "tf5"

# ----- phase toggles --------------------------------------------------------------
RUN_PHASE0  = True
RUN_PHASE1  = False             # heavy (hours); flip on for the real Phase 1 pass
RUN_SWEEPS  = True              # cheap post-hoc sweeps over saved arrays

# ----- model weights: attached as Kaggle *Models* (Add Input -> Models) -----------
# Plain read-only dirs (config.json + shards), already offline. No tar.
MODEL_PATHS = {
    "llada":          "/kaggle/input/models/mengaidev/llada/transformers/8b-base/1",
    "diffusiongemma": "/kaggle/input/models/google/diffusiongemma/transformers/diffusiongemma-26b-a4b-it/1",
}
WEIGHTS_DIR = MODEL_PATHS[MODEL]

# ----- support data: attached as Kaggle *Datasets* --------------------------------
# Kaggle's mount base has varied between /kaggle/input/<slug> and
# /kaggle/input/datasets/<owner>/<slug>; resolve each leaf against both so the
# notebook works regardless of which layout this session gets.
_OWNER = "aarushmahajan"
def _resolve(slug, inner, what):
    for base in (f"/kaggle/input/datasets/{_OWNER}/{slug}", f"/kaggle/input/{slug}"):
        p = os.path.join(base, inner)
        if os.path.exists(p):
            return p
    # not found yet — return the canonical candidate; the mount cell asserts.
    return os.path.join(f"/kaggle/input/datasets/{_OWNER}/{slug}", inner)

SCORER_DIR  = _resolve("autoguidance-gpt2-large", "gpt2_scorer/gpt2_large", "gpt2 scorer")
HFDATA_DIR  = _resolve("autoguidance-hfdata",     "hfdata",                 "hfdata")
MAUVE_DIR   = _resolve("autoguidance-mauve-feat", "mauve/mauve_feat",       "mauve featurizer")
WHEELS_BASE = _resolve("autoguidance-wheels",     "wheels/wheels",          "wheelhouse")
CODE_PKG    = _resolve("autoguidance-code",       "autoguidance",           "code package")

WHEELS_DIR = os.path.join(WHEELS_BASE, f"wheels_{TRANSFORMERS_VARIANT}")

# offline nltk (self-BLEU punkt) ships inside hfdata
NLTK_DATA_DIR = os.path.join(HFDATA_DIR, "nltk_data")
os.environ["NLTK_DATA"] = NLTK_DATA_DIR

# ----- outputs on /kaggle/working (this is what Save Version snapshots) -----------
WORK_ROOT = "/kaggle/working"
# Namespace outputs by MODEL: arrays/verdicts from different models are NOT
# comparable and must never share a dir (evaluate_dir now hard-fails on a mix).
PHASE0_ARRAYS_DIR = os.path.join(WORK_ROOT, "phase0_arrays", MODEL)
PHASE0_OUT = os.path.join(WORK_ROOT, "phase0", MODEL)
PHASE1_OUT = os.path.join(WORK_ROOT, "phase1", MODEL)
for d in (PHASE0_ARRAYS_DIR, PHASE0_OUT, PHASE1_OUT):
    os.makedirs(d, exist_ok=True)

# ----- Phase-0 pass thresholds (editable; the sweep cell reuses these) ------------
THRESHOLDS = {
    "check1_entropy_fraction": 0.80,
    "check2_top1_agreement":   0.60,
    "check2_spearman_rho":     0.50,
    "check3_pearson_r":        0.20,
    "check3_precision":        0.50,
}

print("[config] MODEL =", MODEL, "| variant =", TRANSFORMERS_VARIANT)
print("[config] RUN_PHASE0/1/SWEEPS =", RUN_PHASE0, RUN_PHASE1, RUN_SWEEPS)
print("[config] resolved input dirs:")
for k, v in {"WEIGHTS_DIR": WEIGHTS_DIR, "CODE_PKG": CODE_PKG, "WHEELS_DIR": WHEELS_DIR,
             "HFDATA_DIR": HFDATA_DIR, "SCORER_DIR": SCORER_DIR, "MAUVE_DIR": MAUVE_DIR,
             "NLTK_DATA_DIR": NLTK_DATA_DIR}.items():
    print(f"          {k:14s} -> {v}  (exists={os.path.exists(v)})")
print("[config] outputs: arrays=%s phase0=%s phase1=%s" % (PHASE0_ARRAYS_DIR, PHASE0_OUT, PHASE1_OUT))
print("[config] HF offline:", os.environ["HF_HUB_OFFLINE"], os.environ["TRANSFORMERS_OFFLINE"])


## 2 · Offline pip install (chosen wheelhouse) — RUN FIRST (before any torch import)

In [ ]:
# OFFLINE INSTALL — must run BEFORE any `import torch` in this notebook.
# Pin the exact wheelhouse versions and --force-reinstall so the coherent
# (torch + transformers + accelerate) stack REPLACES Kaggle's base packages.
# Bare/unpinned names get skipped as "already satisfied" and leave a broken mix.
import sys, subprocess

STACK = {
    "tf446": ["torch==2.12.1", "transformers==4.46.3", "tokenizers==0.20.3",
              "accelerate==1.14.0", "safetensors==0.8.0"],
    "tf5":   ["torch==2.12.1", "transformers==5.13.0", "tokenizers==0.22.2",
              "accelerate==1.14.0", "safetensors==0.8.0"],
}[TRANSFORMERS_VARIANT]
EXTRAS = ["sentencepiece", "datasets", "mauve-text", "scikit-learn", "scipy",
          "nltk", "sacrebleu", "einops", "tqdm", "pyyaml"]

cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--force-reinstall",
       "--find-links", WHEELS_DIR] + STACK + EXTRAS
print("[pip] $", " ".join(cmd))
r_ = subprocess.run(cmd, capture_output=True, text=True)
print(r_.stdout[-3500:])
if r_.returncode != 0:
    print("[pip] stderr:\n", r_.stderr[-4000:])
    raise RuntimeError("offline pip install failed — a wheel/dep may be missing from the wheelhouse")

# Kaggle preinstalls torchvision/torchaudio built against its BASE torch. After we
# swap in wheelhouse torch 2.12.1 they mismatch, and transformers.modeling_utils
# imports torchvision -> "torchvision has no attribute 'extension'". We only run
# text models, so remove them; transformers cleanly skips absent vision deps.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "torchvision", "torchaudio"], capture_output=True, text=True)
print("[pip] removed torchvision/torchaudio (mismatched with wheelhouse torch)")

# Verify torch + transformers resolved to the pinned wheelhouse versions.
import importlib, torch, transformers
importlib.reload(transformers)
tv = transformers.__version__
print("[pip] torch:", torch.__version__, "| transformers:", tv)
assert torch.__version__.startswith("2.12"), f"expected torch 2.12.x, got {torch.__version__}"
if TRANSFORMERS_VARIANT == "tf446":
    assert tv.startswith("4.46"), f"variant tf446 expects transformers 4.46.x, got {tv}"
else:
    assert int(tv.split(".")[0]) >= 5, f"variant tf5 expects transformers>=5, got {tv}"
# proves the torch stack is internally consistent (the symbol the base torch lacked)
from torch._utils import _maybe_view_chunk_cat  # noqa: F401
print("[pip] OK — pinned self-consistent stack installed for variant", TRANSFORMERS_VARIANT)


## 3 · GPU info (debug)

In [ ]:
# Show the card + assert we are on the ~96 GB RTX PRO 6000. Warn (not hard-fail) so
# the notebook still runs on a smaller dev GPU for the synthetic dry-run.
import subprocess, torch
subprocess.run(["nvidia-smi"])

assert torch.cuda.is_available(), "CUDA not available — set Accelerator = GPU"
dev = torch.device("cuda:0")
name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\n[gpu] device   : {dev}  ({name})")
print(f"[gpu] total VRAM: {total_gb:.1f} GB")
print(f"[gpu] torch     : {torch.__version__}  bf16 supported: {torch.cuda.is_bf16_supported()}")
if total_gb < 80:
    print(f"[gpu] WARNING: {total_gb:.0f} GB < ~96 GB expected. Real weights may OOM; dry-run still ok.")
else:
    print("[gpu] OK: large-VRAM card detected.")


## 4 · Mount verification

In [ ]:
# List every attached input and assert the leaf dirs this run needs are present.
print("[mount] /kaggle/input contents:")
for d in sorted(os.listdir("/kaggle/input")):
    print("       ", d)

required = {
    "CODE_PKG": CODE_PKG, "WHEELS_DIR": WHEELS_DIR, "HFDATA_DIR": HFDATA_DIR,
    "SCORER_DIR": SCORER_DIR, "MAUVE_DIR": MAUVE_DIR, "WEIGHTS_DIR": WEIGHTS_DIR,
}
missing = [k for k, v in required.items() if not os.path.isdir(v)]
assert not missing, (
    f"missing required inputs: {missing} -> "
    + ", ".join(f"{k}={required[k]}" for k in missing)
    + " (attach the datasets/models)")

# wikitext must be present for Phase 0/1 text loading
wt = os.path.join(HFDATA_DIR, "wikitext-103-v1")
assert os.path.isdir(wt), f"wikitext not found under hfdata: {wt}"

print("\n[mount] du -sh of required inputs:")
for k, v in required.items():
    subprocess.run(["du", "-sh", v])
print("[mount] OK — all required inputs resolved.")


## 5 · Install the `autoguidance` package

In [ ]:
# The code dataset mounts READ-ONLY, so `pip install -e` (develop) cannot write
# its egg-info into the source tree. The package is pure-Python -> just put its
# src/ on sys.path and import.
import sys, os
SRC = os.path.join(CODE_PKG, "src")
assert os.path.isdir(SRC), f"code src not found: {SRC}"
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import autoguidance
from autoguidance.config import Phase0Config, Phase1Config, KaggleConfig
from autoguidance.models import load_adapter
from autoguidance.weak_self import load_weak_self
print("[code] imported autoguidance from", os.path.dirname(autoguidance.__file__))
shipped = os.path.join(SRC, "autoguidance", "SHIPPED_COMMIT.txt")
if os.path.exists(shipped):
    print("[code] shipped commit:", open(shipped).read().strip())


## 6 · Locate the chosen model weights (Kaggle Model mount, no extraction)

In [ ]:
# Weights are attached as a Kaggle Model = a plain read-only directory already on
# disk. Nothing to extract; just point at it and confirm it is populated.
assert os.path.isdir(WEIGHTS_DIR), f"weights dir missing: {WEIGHTS_DIR}"
print("[weights] using Kaggle Model mount (no extraction):", WEIGHTS_DIR)
subprocess.run(["du", "-sh", WEIGHTS_DIR])


## 6b · Weight extraction sanity

In [ ]:
# Assert config.json + weight shards (+ LLaDA remote modeling_*.py) are present.
files = sorted(os.listdir(WEIGHTS_DIR))
print("[sanity] WEIGHTS_DIR files:")
for fn in files:
    print("       ", fn)

assert "config.json" in files, "config.json missing from extracted weights"
shards = [f for f in files if f.endswith(".safetensors") or f.endswith(".bin")]
assert shards, "no weight shards (*.safetensors / *.bin) found"
print(f"[sanity] {len(shards)} weight shard(s) found")

if MODEL == "llada":
    modeling = [f for f in files if f.startswith("modeling_") and f.endswith(".py")]
    assert modeling, "LLaDA needs remote modeling_*.py for trust_remote_code"
    print("[sanity] LLaDA modeling files:", modeling)
print("[sanity] OK.")


## 7 · DRY-RUN — full Phase-0 flow on the synthetic adapter (seconds)

In [ ]:
# Exercise arrays-dump -> evaluate_dir -> verdict end-to-end on the CPU-cheap
# SyntheticAdapter BEFORE touching real weights. Catches contract breaks in seconds.
import tempfile
from autoguidance.phase0.characterize import run_characterization
from autoguidance.phase0.thresholds import evaluate_dir

dry_cfg = Phase0Config()
dry_cfg.device_main = "cpu"          # synthetic runs on CPU
dry_cfg.precision = "bf16"
dry_cfg.n_samples = 8
dry_cfg.batch_size = 4
# MoE-sim so reduced_expert exercises its code path on synthetic.
dry_cfg.constructions = ["input_noise_remask", "input_noise_gauss",
                         "low_nfe", "layer_drop", "inference_dropout", "reduced_expert"]

dry_adapter = load_adapter("synthetic", dry_cfg)
# enable MoE simulation on the synthetic adapter if it exposes the hook
if hasattr(dry_adapter, "enable_moe_sim"):
    dry_adapter.enable_moe_sim(True)
    print("[dry] synthetic MoE-sim enabled for reduced_expert")

dry_weak = {}
for nm in dry_cfg.constructions:
    try:
        dry_weak[nm] = load_weak_self(nm, dry_cfg)
    except NotImplementedError as e:
        print(f"[dry] skip {nm}: {e}")

dry_dir = tempfile.mkdtemp(prefix="phase0_dry_")
print("[dry] arrays_dir:", dry_dir)
paths = run_characterization(dry_adapter, dry_weak, dry_cfg, arrays_dir=dry_dir)
print("[dry] dumped arrays:", {k: os.path.basename(v) for k, v in paths.items()})

verdicts = evaluate_dir(dry_dir, dry_cfg)
print("\n[dry] verdicts:")
for nm, v in verdicts.items():
    print(f"   {nm:22s} all_pass={v.get('all_pass')}  "
          f"c1={v['check1']['check1_pass']} c2={v['check2']['check2_pass']} c3={v['check3']['check3_pass']}")
print("[dry] OK — contract end-to-end works. Proceeding to real weights.")


## 8 · VRAM probe + load the real adapter (bf16, cuda:0)

In [ ]:
# Build the real Phase-0 config pointing at the extracted weights, then load the
# adapter in bf16 on cuda:0. Print VRAM before/after.
import torch, gc

def vram():
    free, total = torch.cuda.mem_get_info(0)
    return (total - free) / 1e9

cfg = Phase0Config()
cfg.device_main = "cuda:0"
cfg.device_eval = "cuda:0"
cfg.precision = "bf16"
cfg.dtype = "bfloat16"
cfg.batch_size = 8
cfg.max_seq_len = 256
cfg.transformers_variant = TRANSFORMERS_VARIANT
cfg.n_samples = 200
cfg.constructions = ["input_noise_remask", "input_noise_gauss", "low_nfe",
                     "layer_drop", "inference_dropout", "reduced_expert"]
# tell the adapter where the local weights live (offline)
cfg.model_path = WEIGHTS_DIR
cfg.hfdata_path = HFDATA_DIR   # offline wikitext via load_from_disk
for k, v in THRESHOLDS.items():
    setattr(cfg, k, v)

print(f"[load] VRAM before: {vram():.1f} GB")
gc.collect(); torch.cuda.empty_cache()
adapter = load_adapter(MODEL, cfg)
print(f"[load] VRAM after : {vram():.1f} GB")
print(f"[load] adapter={type(adapter).__name__} device={adapter.device} "
      f"vocab={adapter.vocab_size} mask_id={adapter.mask_token_id} "
      f"n_layers={adapter.n_layers} is_moe={adapter.is_moe}")


## 9 · MoE patch confirm (DiffusionGemma only)

In [ ]:
# For the MoE model, confirm the reduced-expert patch actually FIRES on the real
# weights (the only place these can be checked). Two build-time diagnostics:
#   1. per-block config identity — catches the "block cached its own config copy,
#      so mutating model.config is a silent no-op" risk.
#   2. full-vs-reduced logits must DIFFER — proves top_k reduction changed routing.
if MODEL == "diffusiongemma" and adapter.is_moe:
    from autoguidance.weak_self.moe_patch import list_moe_modules, reduce_moe_topk
    print("[moe] router / top-k attributes found:")
    list_moe_modules(adapter._model)

    # (1) does every submodule that carries a .config share model.config's object?
    root_cfg = getattr(adapter._model, "config", None)
    shared = mismatched = 0
    for m in adapter._model.modules():
        mc = getattr(m, "config", None)
        if mc is None:
            continue
        if mc is root_cfg:
            shared += 1
        else:
            mismatched += 1
    print(f"[moe] submodule .config identity vs root: shared={shared} mismatched={mismatched}")
    if mismatched:
        print("[moe] WARNING: some blocks hold a NON-root config object — a config-level "
              "top_k patch may not reach them. Inspect before trusting reduced_expert.")

    # (2) prove the patch fires: reduced routing must change the logits.
    txt = ["The quick brown fox jumps over the lazy dog."] * 2
    x = torch.cat([adapter.encode(t) for t in txt], dim=0).to(adapter.device)
    with torch.no_grad():
        full = adapter.logits(x)
        with reduce_moe_topk(adapter._model, cfg.reduced_expert_topk):
            red = adapter.logits(x)
    diff = (full.float() - red.float()).abs().max().item()
    agree = (full.argmax(-1) == red.argmax(-1)).float().mean().item()
    print(f"[moe] max|full-reduced| logit diff: {diff:.4f}  (argmax agreement {agree:.3f})")
    assert diff > 0, ("[moe] reduced pass IDENTICAL to full — top_k patch did NOT fire. "
                      "reduced_expert would equal the full model. Fix moe_patch before running.")
    print("[moe] OK — patch fires (reduced routing differs from full).")
else:
    print("[moe] skipped (not a MoE model / MODEL != diffusiongemma).")


## 10 · Phase 0 — characterize (2 forward passes/sample, dump arrays)

In [ ]:
# Real Phase-0 characterization: dumps the 7 per-construction arrays to
# PHASE0_ARRAYS_DIR. No pass/fail here — thresholding is the next (cheap) cell.
if RUN_PHASE0:
    from autoguidance.phase0.characterize import run_characterization
    weak_selfs = {}
    for nm in cfg.constructions:
        try:
            weak_selfs[nm] = load_weak_self(nm, cfg)
        except NotImplementedError as e:
            print(f"[phase0] skip {nm}: {e}")
    print("[phase0] running constructions:", list(weak_selfs))
    paths = run_characterization(adapter, weak_selfs, cfg, arrays_dir=PHASE0_ARRAYS_DIR)
    print("\n[phase0] dumped arrays:")
    for nm, p in paths.items():
        print(f"   {nm:22s} -> {p}")
else:
    print("[phase0] RUN_PHASE0 == False — skipping characterization.")


## 11 · Threshold + sweep (CHEAP, re-runnable, no model reload)

In [ ]:
# Reuses the saved arrays only — edit THRESHOLDS above and re-run this cell freely.
if RUN_PHASE0:
    from autoguidance.phase0.thresholds import evaluate_dir
    from autoguidance.phase0.sweep import sweep_threshold, sweep_construction_param

    # apply current thresholds to cfg
    for k, v in THRESHOLDS.items():
        setattr(cfg, k, v)

    print("[eval] verdicts at current thresholds:")
    verdicts = evaluate_dir(PHASE0_ARRAYS_DIR, cfg)
    print(f"{'construction':22s} {'c1':>4s} {'c2':>4s} {'c3':>4s} {'ALL':>5s}")
    for nm, vv in verdicts.items():
        print(f"{nm:22s} "
              f"{str(vv['check1']['check1_pass']):>4s} "
              f"{str(vv['check2']['check2_pass']):>4s} "
              f"{str(vv['check3']['check3_pass']):>4s} "
              f"{str(vv.get('all_pass')):>5s}")

    if RUN_SWEEPS:
        # (a) threshold sweep — pure numpy over saved arrays
        grid = {
            "check2_top1_agreement": [0.50, 0.55, 0.60, 0.65, 0.70],
            "check3_pearson_r":      [0.10, 0.15, 0.20, 0.25, 0.30],
        }
        print("\n[sweep] threshold sweep grid:", grid)
        rows = sweep_threshold(PHASE0_ARRAYS_DIR, grid, cfg)
        print(f"[sweep] {len(rows)} rows; sample:")
        for row in rows[:12]:
            print("   ", row)

        # (b) per-construction parameter sweeps — re-run ONLY the weak pass, cache full.
        full_cache = {}
        param_grids = {
            "noise_rate_remask": cfg.sweep_remask_rate,
            "noise_sigma_gauss": cfg.sweep_gauss_sigma,
            "layer_drop_k":      cfg.sweep_layer_drop_k,
            "dropout_p":         cfg.sweep_dropout_p,
        }
        for pname, values in param_grids.items():
            sweep_dir = os.path.join(WORK_ROOT, f"phase0_sweep_{pname}")
            os.makedirs(sweep_dir, exist_ok=True)
            print(f"\n[sweep] construction-param {pname} over {values} -> {sweep_dir}")
            try:
                out = sweep_construction_param(adapter, cfg, pname, values,
                                               sweep_dir, full_cache=full_cache)
                for val, npz in out.items():
                    print(f"   {pname}={val} -> {os.path.basename(npz)}")
            except NotImplementedError as e:
                print(f"   skip {pname}: {e}")
else:
    print("[eval] RUN_PHASE0 == False — skipping threshold/sweep.")


## 12 · Phase 0 report + preview

In [ ]:
# Render the verdict dict to report artifacts under /kaggle/working/phase0, then
# preview the markdown table + verdict text + the two PNGs inline.
if RUN_PHASE0:
    from autoguidance.phase0.report import save_results
    save_results(verdicts, PHASE0_OUT)

    print("\n===== agreement_table.md =====")
    print(open(os.path.join(PHASE0_OUT, "agreement_table.md")).read())
    print("===== verdict.txt =====")
    print(open(os.path.join(PHASE0_OUT, "verdict.txt")).read())

    from IPython.display import Image, display
    for png in ("entropy_dist.png", "error_position_scatter.png"):
        p = os.path.join(PHASE0_OUT, png)
        if os.path.exists(p):
            print("\n[report]", png)
            display(Image(filename=p))
else:
    print("[report] RUN_PHASE0 == False — skipping report.")


## 12b · Preview model outputs (decoded predictions per construction)


In [ ]:
# Concrete look at what the model actually predicts, straight from the saved arrays
# (gt / top1_full / top1_weak) — no model reload. Shows ground-truth vs full vs weak
# top-1 token at masked positions, so the entropy numbers become readable text.
if RUN_PHASE0:
    import numpy as np
    from autoguidance.phase0.arrays import load_arrays, list_constructions

    def _tok(i):
        try:
            return repr(adapter.decode(torch.tensor([[int(i)]], device=adapter.device)))
        except Exception:
            return f"<{int(i)}>"

    for name in list_constructions(PHASE0_ARRAYS_DIR):
        a = load_arrays(PHASE0_ARRAYS_DIR, name)
        gt, tf, tw = a["gt"], a["top1_full"], a["top1_weak"]
        ef, ew = a["entropy_full"], a["entropy_weak"]
        n = min(12, len(gt))
        print(f"\n=== {name}  (showing {n}/{len(gt)} masked positions) ===")
        print(f"{'gt':>18} | {'full_top1':>18} | {'weak_top1':>18} | H_full  H_weak  match")
        for i in range(n):
            print(f"{_tok(gt[i]):>18} | {_tok(tf[i]):>18} | {_tok(tw[i]):>18} | "
                  f"{ef[i]:6.3f}  {ew[i]:6.3f}   {'=' if tf[i]==tw[i] else 'x'}")
        print(f"  full==gt: {(tf==gt).mean():.3f}   weak==gt: {(tw==gt).mean():.3f}   "
              f"full==weak: {(tf==tw).mean():.3f}   median H_full: {np.median(ef):.2e}")
else:
    print("[preview] RUN_PHASE0 == False — nothing to preview.")


## 13 · Phase 1 — baselines (optional, heavy)

In [ ]:
# Decoders × NFE ladder over the same real adapter. Hours-long for the full ladder.
if RUN_PHASE1:
    from autoguidance.phase1.baseline_runner import run_baselines
    from autoguidance.phase1.report import save_results as save_phase1

    p1 = Phase1Config()
    p1.device_main = "cuda:0"; p1.device_eval = "cuda:0"
    p1.precision = "bf16"; p1.dtype = "bfloat16"
    p1.batch_size = 8; p1.max_seq_len = 256
    p1.transformers_variant = TRANSFORMERS_VARIANT
    p1.model_path = WEIGHTS_DIR
    p1.hfdata_path = HFDATA_DIR
    p1.scorer_model = "gpt2-large"
    # point eval at offline gpt2-large + mauve mounts
    p1.scorer_path = SCORER_DIR
    p1.mauve_path = MAUVE_DIR
    print("[phase1] decoders=%s nfe=%s n=%d" % (p1.decoders, p1.nfe_ladder, p1.n_samples))

    rows = run_baselines(adapter, p1)
    save_phase1(rows, PHASE1_OUT)
    print("[phase1] wrote ->", PHASE1_OUT)
    print(open(os.path.join(PHASE1_OUT, "results.csv")).read())
else:
    print("[phase1] RUN_PHASE1 == False — skipping baselines.")


## 14 · Egress

In [ ]:
# PRIMARY EGRESS: "Save Version" snapshots all of /kaggle/working (arrays, phase0,
# phase1). That is the recommended offline-safe export — no network needed.
print("[egress] /kaggle/working tree (this is what Save Version captures):")
subprocess.run(["du", "-sh", PHASE0_ARRAYS_DIR, PHASE0_OUT, PHASE1_OUT])
for d in (PHASE0_OUT, PHASE1_OUT):
    if os.path.isdir(d):
        print(f"\n[egress] {d}:")
        for fn in sorted(os.listdir(d)):
            print("       ", fn)

print("""
[egress] OPTIONAL networked export (needs Internet = ON, defeats offline guarantee):
  import shutil, subprocess
  res = "/kaggle/working/results_pack"; shutil.copytree(PHASE0_OUT, res+"/phase0")
  # write dataset-metadata.json {id: <user>/autoguidance-results, licenses CC0-1.0}
  # kaggle datasets create -p res   (or: kaggle datasets version -p res -m msg)
Primary path = Save Version snapshot of /kaggle/working. Done.
""")
